In [1]:
from pathlib import Path
from dotenv import load_dotenv
import os

ROOT = Path.cwd().parent

env_path = ROOT / ".env.backtest"

load_dotenv(env_path)

print(env_path)
print(os.getenv("MONGO_HOST"))

os.chdir(Path.cwd().parent)

/Users/paulvogt/mongo_db_streamlit/.env.backtest
localhost


In [2]:
%pwd

'/Users/paulvogt/mongo_db_streamlit'

In [3]:
from infrastructure.mongo.mongo_repository import MongoRepository
from config.mongo_config import MongoCollection, MongoDatabase, MongoUser
from infrastructure.mongo.mongo_connection import MongoConnection
from core.application.notebook_service import NotebookService

In [4]:
with MongoConnection(user=MongoUser.DASHBOARDUSER) as conn:
    finance_repo = MongoRepository(conn, MongoDatabase.PROCESSED, MongoCollection.FINANCEDATA)
    company_repo = MongoRepository(conn, MongoDatabase.PROCESSED, MongoCollection.COMPANYDATA)
    sector_repo = MongoRepository(conn,  MongoDatabase.PROCESSED, MongoCollection.SECTORDATA)
    sp_500_repo = MongoRepository(conn,  MongoDatabase.PROCESSED, MongoCollection.SP500)
    constituents_repo = MongoRepository(conn,  MongoDatabase.PROCESSED, MongoCollection.SCD_CONSTITUENTS)
    
    notebook_service = NotebookService(finance_repo, company_repo, sector_repo, sp_500_repo, constituents_repo)

    df = notebook_service.run()
    df_benchmark = notebook_service.getIndexData()
    df_benchmark.head()
    

/Users/paulvogt/mongo_db_streamlit/core/application/notebook_service.py:62: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_result = pd.concat([df_result, df_grouped[res_cols]])
Less than 10 Companies in Communication Services on 2009-12-31 00:00:00
Less than 10 Companies in Communication Services on 2011-12-31 00:00:00
Less than 10 Companies in Real Estate on 2008-12-31 00:00:00
Less than 10 Companies in Real Estate on 2011-12-31 00:00:00
Less than 10 Companies in Utilities on 2022-12-31 00:00:00
Less than 10 Companies in Utilities on 2023-12-31 00:00:00
Less than 10 Companies in Utilities on 2024-12-31 00:00:00
/Users/paulvogt/mongo_db_streamlit/core/application/notebook_service.py:100: FutureWarning: The behavior of DataFrame concatenat

In [5]:
#for (sector, strategy), group in df.groupby(["sector", "strategy"]):
#
#    triangle = group.pivot_table(
#        index="buyyear",
#        columns="sellyear",
#        values="rendite"
#    ).sort_index().sort_index(axis=1)
#
#    print("\n" + "="*60)
#    print(f"{sector} | {strategy}")
#    print("="*60)
#
#    print(triangle.round(2))

In [6]:
df.to_csv("backtest/data/backtest_data.csv", index=False)
df_benchmark.to_csv("backtest/data/benchmark_data.csv", index=False)